# Train all five pet segmentation models on a Kaggle GPU

**One-time setup** (right-hand panel of the notebook editor):
1. **Session options → Accelerator: GPU T4 x2.** Avoid P100: recent PyTorch builds no longer support it.
2. **Session options → Internet: On** (needs a phone-verified Kaggle account).
3. **Add-ons → Secrets**, then attach the ones you need to this notebook:
   - `HF_TOKEN`: a Hugging Face read token. Required for `dino_linear`: the DINOv3 weights are gated, so also accept the license on the model page first.
   - `GITHUB_TOKEN`: only if your repo is private.
4. **Add Input** → an Oxford-IIIT Pet dataset (recommended). Any mirror works, whatever its folder layout. Missing trimaps or split files are filled in from the official annotations (19 MB). With no input, the full official dataset is downloaded (about 800 MB).

**Running**
- Set `EPOCHS = 1` below and run all cells once interactively as a smoke test (about 15 minutes).
- Then set `EPOCHS = 30` and use **Save Version → Save & Run All (Commit)**. It runs in the background (up to 12 hours, roughly 3 to 4 needed here), and you can close the tab.
- When it finishes, open the version's **Output** tab and download `pet-seg-outputs.zip`.

## 1. Settings

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/<your-username>/pet-seg.git"   # your GitHub repo
BRANCH = "main"

MODELS = ["unet_scratch", "unet_resnet34", "deeplabv3plus", "segformer", "dino_linear"]
EPOCHS = 30            # 1 for a quick smoke test
BATCH_SIZE = 16
NUM_WORKERS = 4        # Kaggle gives 4 CPU cores. Lowered automatically if shared memory is small.
EXTRA_SET = ""         # extra overrides for every model, e.g. "data.image_size=192"
KEEP_LAST_CKPT = False # also put last.pt (resumable state, about 3x bigger) in the download zip

# To resume a run that died: Add Input -> this notebook's previous version, then point here, e.g.
# PREVIOUS_RUNS = Path("/kaggle/input/<notebook-slug>/pet-seg/runs")
PREVIOUS_RUNS = None

WORK = Path("/kaggle/working")        # everything here is saved as notebook output
REPO = WORK / "pet-seg"
DATA_PARENT = Path("/tmp/pet-data")   # outside /kaggle/working so the dataset is not saved as output
KAGGLE_INPUT = Path("/kaggle/input")

## 2. Check the GPU

In [ ]:
import shutil, subprocess
import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    major, minor = torch.cuda.get_device_capability(0)
    if (major, minor) < (7, 0):
        print("WARNING: this GPU is too old for current PyTorch builds. Switch the accelerator to GPU T4 x2.")
else:
    print("WARNING: no GPU. Session options -> Accelerator -> GPU T4 x2, then restart the session.")

shm_gb = shutil.disk_usage("/dev/shm").total / 1e9
print(f"shared memory: {shm_gb:.1f} GB")
if shm_gb < 1 and NUM_WORKERS > 0:
    NUM_WORKERS = 0
    print("small /dev/shm, using NUM_WORKERS = 0 to avoid DataLoader 'bus error' crashes")

## 3. Get the code
Clones your repo. Tokens come from Kaggle Secrets and are never written into the saved output.

In [ ]:
import os

def secret(name):
    """A Kaggle secret attached to this notebook, else an environment variable, else None."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

gh = secret("GITHUB_TOKEN")
auth_url = REPO_URL.replace("https://", f"https://x-access-token:{gh}@") if gh and REPO_URL.startswith("https://") else REPO_URL

def git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError((r.stderr or r.stdout).replace(gh or "\0", "***"))
    return r.stdout

if (REPO / ".git").exists():
    git("pull", "--ff-only", auth_url, BRANCH, cwd=REPO)        # pull without storing the token
else:
    git("clone", "--depth", "1", "-b", BRANCH, auth_url, str(REPO))
    git("remote", "set-url", "origin", REPO_URL, cwd=REPO)      # keep the token out of .git/config
print(git("log", "-1", "--format=%h %s", cwd=REPO))

hf = secret("HF_TOKEN")
if hf:
    os.environ["HF_TOKEN"] = hf     # inherited by the training processes
print("HF_TOKEN:", "set" if hf else "missing, dino_linear will fail to download DINOv3")

os.chdir(REPO)
import sys
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

## 4. Install packages
Kaggle already has PyTorch with CUDA. This only adds or upgrades what the repo needs on top, and never reinstalls PyTorch.

In [ ]:
!pip install -q "albumentations>=2.0" "segmentation-models-pytorch>=0.4" "transformers>=4.56" "timm>=1.0.20"
import albumentations, segmentation_models_pytorch, transformers, timm
print("albumentations", albumentations.__version__, "| smp", segmentation_models_pytorch.__version__,
      "| transformers", transformers.__version__, "| timm", timm.__version__)

## 5. Find the dataset
Uses the Oxford-IIIT Pet dataset you added as input, whatever its folder layout: it locates the image folder,
the trimaps and the official split files, and links them into the layout the code expects (no copying).
If the input lacks trimaps or split files, only the official annotations archive (about 19 MB) is downloaded.
With no input at all, the full official dataset is downloaded (about 800 MB).

In [ ]:
import os
from fnmatch import fnmatch
from torchvision.datasets import OxfordIIITPet
from torchvision.datasets.utils import download_and_extract_archive

def busiest_dir(base, pattern, name=None, min_files=50):
    """The folder under base holding the most files matching pattern (optionally only folders called name)."""
    best, n_best = None, 0
    if not Path(base).exists():
        return None
    for d, _, files in os.walk(base, followlinks=True):
        if name and Path(d).name != name:
            continue
        n = sum(1 for f in files if fnmatch(f.lower(), pattern) and not f.startswith("._"))
        if n > n_best:
            best, n_best = Path(d), n
    return best if n_best >= min_files else None

def find_splits(base):
    """The folder under base containing both official split files."""
    if not Path(base).exists():
        return None
    for d, _, files in os.walk(base, followlinks=True):
        if "trainval.txt" in files and "test.txt" in files:
            return Path(d)
    return None

images = busiest_dir(KAGGLE_INPUT, "*.jpg")
trimaps = busiest_dir(KAGGLE_INPUT, "*.png", name="trimaps")
split_dir = find_splits(KAGGLE_INPUT)
print("from inputs -> images:", images, "| trimaps:", trimaps, "| split files:", split_dir)

if images and not (trimaps and split_dir):
    print("input is missing trimaps or split files, fetching the official annotations (about 19 MB) ...")
    url, md5 = OxfordIIITPet._RESOURCES[1]
    official = DATA_PARENT / "official"
    if not (official / "annotations" / "trimaps").exists():
        download_and_extract_archive(url, download_root=str(official), md5=md5)
    trimaps = trimaps or official / "annotations" / "trimaps"
    split_dir = split_dir or official / "annotations"

if images:
    DATA_ROOT = DATA_PARENT / "oxford-iiit-pet"
    ann = DATA_ROOT / "annotations"
    ann.mkdir(parents=True, exist_ok=True)
    for link, target in ((DATA_ROOT / "images", images), (ann / "trimaps", trimaps),
                         (ann / "trainval.txt", split_dir / "trainval.txt"), (ann / "test.txt", split_dir / "test.txt")):
        if link.is_symlink() or link.exists():
            link.unlink()
        link.symlink_to(target)
else:
    print("no Oxford-IIIT Pet images in the inputs, downloading the full official dataset ...")
    OxfordIIITPet(root=str(DATA_PARENT), split="trainval", target_types="segmentation", download=True)
    DATA_ROOT = DATA_PARENT / "oxford-iiit-pet"

# Every listed image must have its picture and trimap, or the mirror uses different file names.
from src.dataset import make_splits
splits = make_splits(DATA_ROOT)
records = [r for v in splits.values() for r in v]
missing = [r.name for r in records if not (DATA_ROOT / "images" / f"{r.name}.jpg").exists()
           or not (DATA_ROOT / "annotations" / "trimaps" / f"{r.name}.png").exists()]
print("dataset root:", DATA_ROOT)
print({k: len(v) for k, v in splits.items()}, f"| {len(missing)} listed images missing a file")
if len(missing) > 0.05 * len(records):
    raise RuntimeError(f"{len(missing)} of {len(records)} images or trimaps not found (e.g. {missing[:3]}). "
                       "The input mirror probably renames files. Remove it from the inputs to use the official download.")

## 6. Sanity checks
Unit tests (about 20 s), then one sample and its augmentations. The outline must sit on the pet in every panel.

In [ ]:
!python -m pytest -q tests

%matplotlib inline
from src.dataset import OxfordPetSeg
from src import viz
ds = OxfordPetSeg(DATA_ROOT, split="train")
viz.show_sample(ds, 0)
viz.show_augmented(ds, 0, n=8);

## 7. Train
Each model trains in its own process, so one failure (for example a missing `HF_TOKEN`) does not stop the others.

- A run with `last.pt` but no `summary.json` was interrupted: it **resumes**.
- A finished run with fewer epochs than `EPOCHS` (your smoke test) is **restarted** from scratch.
- A finished run with at least `EPOCHS` epochs is **skipped**.

In [ ]:
import json, time
from IPython import get_ipython

if PREVIOUS_RUNS:
    shutil.copytree(PREVIOUS_RUNS, REPO / "runs", dirs_exist_ok=True)
    print("restored runs from", PREVIOUS_RUNS)

status = {}
for m in MODELS:
    run = REPO / "runs" / m
    summary = run / "summary.json"
    sets = (f"data.root={DATA_ROOT} train.epochs={EPOCHS} train.batch_size={BATCH_SIZE} "
            f"data.num_workers={NUM_WORKERS} log.print_every=50 {EXTRA_SET}").strip()
    if summary.exists():
        done = json.loads(summary.read_text()).get("epochs", 0)
        if done >= EPOCHS:
            print(f"=== {m}: already trained for {done} epochs, skipping")
            status[m] = "skipped (done)"
            continue
        print(f"=== {m}: finished a {done}-epoch run earlier, starting over for {EPOCHS} epochs")
        shutil.rmtree(run)
    resume = (run / "last.pt").exists()
    cmd = f"python src/train.py --config configs/{m}.yaml {'--resume ' if resume else ''}--set {sets}"
    print(f"=== {m}: {'resuming' if resume else 'training'}\n$ {cmd}", flush=True)
    t0 = time.time()
    get_ipython().system(cmd)
    code_ = get_ipython().user_ns.get("_exit_code", 0)
    status[m] = f"{'ok' if code_ == 0 else f'FAILED (exit {code_})'} in {(time.time() - t0) / 60:.1f} min"

print("\n".join(f"{m:15s} {s}" for m, s in status.items()))

## 8. Training curves and the last validation overlays

In [ ]:
from IPython.display import Image, display

histories = {m: REPO / "runs" / m / "history.jsonl" for m in MODELS if (REPO / "runs" / m / "history.jsonl").exists()}
if histories:
    viz.plot_training_curves(histories)
for m in histories:
    overlays = sorted((REPO / "runs" / m / "overlays").glob("step_*.png"))
    if overlays:
        print(m, overlays[-1].name)
        display(Image(filename=str(overlays[-1]), width=360))

## 9. Evaluate on validation and test
Writes every evaluation chart to `results/val/` and `results/test/`.

In [ ]:
!python src/evaluate.py --split val
!python src/evaluate.py --split test

import pandas as pd
summary = json.loads((REPO / "results" / "test" / "summary.json").read_text())
cols = ["dice", "iou", "pixel_acc", "best_threshold", "dice_at_best_threshold", "boundary_error_share_near",
        "latency_ms", "params_m"]
table = pd.DataFrame(summary).T[cols].astype(float).round(4)
display(table)
for f in ["model_grid_error.png", "training_curves.png", "threshold_sweep.png", "boundary.png", "per_breed.png",
          "efficiency.png"]:
    p = REPO / "results" / "test" / f
    if p.exists():
        print(f)
        display(Image(filename=str(p), width=900))

## 10. Build the HTML gallery and package the outputs

In [ ]:
!python src/viz.py --root {DATA_ROOT} --eval-split test --out results/gallery

import zipfile
zip_path = WORK / "pet-seg-outputs.zip"
keep = lambda p: p.is_file() and (KEEP_LAST_CKPT or p.name not in ("last.pt", "last.pt.tmp"))
files = [p for d in ("runs", "results") for p in sorted((REPO / d).rglob("*")) if keep(p)]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.relative_to(REPO))
print(f"{zip_path}: {len(files)} files, {zip_path.stat().st_size / 1e6:.0f} MB")
for m in MODELS:
    best = REPO / "runs" / m / "best.pt"
    print(f"  {m:15s} {'best.pt included' if best.exists() else 'NO CHECKPOINT (training failed?)'}")

os.chdir(WORK)
from IPython.display import FileLink
display(FileLink("pet-seg-outputs.zip"))

## Using the results on your machine

Download `pet-seg-outputs.zip` (Output tab of the saved version, or the link above in an interactive session), then from the folder that contains `pet-seg/`:

```bash
unzip -o pet-seg-outputs.zip -d pet-seg/
cd pet-seg
python app.py                         # Gradio demo with all five models
python src/viz.py --open              # HTML gallery of every view
python predict.py --model all --input some_pet.jpg --out grid.png
```

The zip unpacks to `runs/` (checkpoints, training history, overlays) and `results/` (metrics and charts). The checkpoints remember the Kaggle dataset path, but the code falls back to `data/oxford-iiit-pet` on your machine automatically.

With the Kaggle CLI (`pip install kaggle`, token in `~/.kaggle/kaggle.json`) you can also download from the terminal: `kaggle kernels output <username>/<notebook-slug> -p .`